# Histology → `hist.parquet` (UNI2-h, Colab **CPU** is fine)

Downloads the 10 gated UNI2-h archives one at a time, averages each slide's tile features (as the
authors' `embed_hist_uni2.py` does), deletes the archive, then averages slides per patient into
`hist.parquet`.

**Needs your own** approved access to `MahmoodLab/UNI2-h-features` and a token saved in Colab
Secrets (key icon, left sidebar) named `HF_TOKEN`, with notebook access switched on. Never paste the
token anywhere.

**Licence:** UNI2-h features are CC BY-NC-ND 4.0 and the terms say not to distribute them. The
output stays in your Drive; do not share it or commit it. Other users with their own access
can run this same notebook to get the same file (up to float rounding).

Outputs go to the Google Drive folder set by `PRIVATE` in cell 1 (default
`My Drive/tcga-multimodal-embeddings`). Change it there to use another folder.

This can run at the same time as the GPU notebook, in a second browser tab. If Colab won't give you
a second runtime, run it afterwards.

**1. Setup.** Expected last lines: `TOTAL 873 306 567 ...` and
`clinical rows: 884 | study patients with a fold: 873`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, zipfile
PRIVATE = '/content/drive/MyDrive/tcga-multimodal-embeddings'
STAGE = f'{PRIVATE}/tcga10'                      # survives disconnects
WORK = '/content/work'                           # local scratch
AUTH = '/content/multimodal-cancer-modeling'     # authors' public repo
TOOLS = '/content/tcga-multimodal-embeddings'
TOOLS_URL = 'https://github.com/YaseenM2/tcga-multimodal-embeddings.git'
for d in (STAGE, WORK):
    os.makedirs(d, exist_ok=True)
if not os.path.isdir(AUTH):
    subprocess.run(['git', 'clone', '-q', '--depth', '1',
                    'https://github.com/StevenSong/multimodal-cancer-modeling.git', AUTH], check=True)
if not os.path.isdir(TOOLS):
    if subprocess.run(['git', 'clone', '-q', TOOLS_URL, TOOLS]).returncode != 0:
        # Repo not published yet: upload tcga-multimodal-embeddings.zip to /content first.
        zipfile.ZipFile('/content/tcga-multimodal-embeddings.zip').extractall('/content')
%cd {TOOLS}
!pip -q install -r requirements.txt
!python scripts/select_cohort.py --clinical {AUTH}/data/clinical.csv --splits {AUTH}/results/split_cases.csv --out-dir {WORK} | tail -3
CASES = f'{WORK}/split_cases.csv'                # 873 case_ids (default 10 projects)

**2. Token and disk.** Expected: a `df` line with tens of GB free (one archive is at most ~10 GB).

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')    # not printed
HIST_ROOT = f'{PRIVATE}/tcga10_hist'                 # slide vectors on Drive: resumable
!df -h /content | tail -1

**3. Pilot on the smallest project (CHOL).** Expected: `TCGA-CHOL: {'matched': ..., 'new': ...,
'missing_or_invalid': 0, ...}` and `UNI2-H PROJECTS COMPLETE!`.

In [ ]:
!python scripts/run_uni2_projects.py --projects CHOL --splits {CASES} \
    --out-root {HIST_ROOT} --download-dir /content/uni2_tmp

**4. All ten projects** (CHOL is skipped; takes 1–2 hours, mostly download). After a disconnect,
run cells 1 and 2 again, then this one: finished projects are skipped.
Expected: one summary line per project, then `UNI2-H PROJECTS COMPLETE!`.

In [ ]:
!python scripts/run_uni2_projects.py --splits {CASES} \
    --out-root {HIST_ROOT} --download-dir /content/uni2_tmp

**5. Patient-level file.** Expected: `"cases_written": 873`, `"cases_missing": 0`, about 6 MiB.
The file is `tcga10/hist.parquet` in the Drive folder from cell 1. If `cases_missing` isn't 0,
the summary says so and no file is written.

In [ ]:
!python scripts/export_parquet.py --modality hist --npy-root {HIST_ROOT} --cases {CASES} \
    --output {STAGE}/hist.parquet \
    --note "UNI2-h tile features, mean per slide (float64 accumulation), mean over slides"